# Project FORESIGHT — Phase 6: Feature Engineering

**Input:** `data/processed/integrated/forecast_base.parquet` (CAM)  
**Output:** `data/processed/features/forecast_features.parquet`  
**Target:** `units_sold`  
**Grain:** `date + source_dataset + entity_id + product_key`  

Strict UCI / SYNTHETIC separation. Leakage-safe lags & rolling windows.  
**Do not proceed to Phase 7 until validation passes.**


## 1. Setup & schema validation

In [1]:
import os, sys
import numpy as np
import pandas as pd

BASE_DIR = os.path.abspath(".")
if os.path.basename(BASE_DIR) == "notebooks":
    BASE_DIR = os.path.abspath("..")
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

from src.feature_engineering import (
    load_features_input,
    validate_forecast_base_schema,
    run_feature_pipeline,
    write_feature_engineering_report,
    get_split_summary,
    get_ml_feature_compatibility,
)
from src.validate_features import run_validation
from src.feature_adapter import get_compatibility_summary

print("BASE_DIR:", BASE_DIR)
fb = load_features_input()
print("forecast_base shape:", fb.shape)
print("columns:", list(fb.columns))
print("sources:\n", fb["source_dataset"].value_counts())
print("date range:", fb["date"].min(), "->", fb["date"].max())
print("Schema validation: PASS")


BASE_DIR: C:\Users\SURAG\Documents\zidio\Project_FORESIGHT\Demand-Inventory-Intelligence
forecast_base shape: (1995496, 12)
columns: ['date', 'source_dataset', 'entity_id', 'entity_type', 'product_key', 'sku_id', 'units_sold', 'revenue', 'average_unit_price', 'transaction_count', 'unique_customers', 'promotion_flag']
sources:
 source_dataset
SYNTHETIC    1461000
UCI           534496
Name: count, dtype: int64[pyarrow]
date range: 2009-12-01 00:00:00 -> 2025-12-31 00:00:00
Schema validation: PASS


## 2. Run full Phase 6 feature pipeline

In [2]:
df, meta = run_feature_pipeline(save=True)
print("output shape:", df.shape)
print("output columns:", list(df.columns))
print(get_split_summary(df).to_string(index=False))


[Phase 6] Loading data...
  forecast_base: 1,995,496 rows × 12 cols
  Schema validation: PASS


[Phase 6] Creating calendar features...


[Phase 6] Creating cyclical features...


[Phase 6] Creating lag features...


[Phase 6] Creating rolling features (leakage-safe)...


...

[Phase 6] Creating chronological time splits...


[Phase 6] Documenting missing values...


[Phase 6] Saved: C:\Users\SURAG\Documents\zidio\Project_FORESIGHT\Demand-Inventory-Intelligence\data\processed\features\forecast_features.parquet
  Output: 1,995,496 rows × 62 cols
[Phase 6] Writing feature registry...
[Phase 6] Writing feature quality report...


output shape: (1995496, 62)
output columns: ['date', 'source_dataset', 'entity_id', 'entity_type', 'product_key', 'sku_id', 'units_sold', 'revenue', 'average_unit_price', 'transaction_count', 'unique_customers', 'promotion_flag', 'year', 'month', 'quarter', 'week_of_year', 'day_of_week', 'day_of_month', 'day_of_year', 'is_weekend', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'units_sold_lag_1', 'units_sold_lag_2', 'units_sold_lag_3', 'units_sold_lag_7', 'units_sold_lag_14', 'units_sold_lag_21', 'units_sold_lag_28', 'units_sold_lag_30', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14', 'rolling_mean_30', 'rolling_std_30', 'demand_change_1', 'demand_change_7', 'demand_growth_7', 'demand_growth_30', 'base_price', 'discount_pct', 'price_lag_1', 'price_change', 'promotion_available', 'promo_rolling_7', 'category', 'sub_category', 'brand', 'region', 'store_type', 'store_size_sqft', 'ending_inventory', 'on_order_qty', 'stockout_flag', 'historical_doi', 'is_holiday', 

source_dataset      split start_date   end_date    rows  unique_entities  unique_products
     SYNTHETIC      train 2022-01-01 2025-03-13 1168000               10              100
     SYNTHETIC validation 2025-03-14 2025-08-06  146000               10              100
     SYNTHETIC       test 2025-08-07 2025-12-31  147000               10              100
           UCI      train 2009-12-01 2011-07-13  401604                1             4657
           UCI validation 2011-07-14 2011-09-25   53174                1             3036
           UCI       test 2011-09-26 2011-12-09   79718                1             3195


## 3. Feature groups & sample rows

In [3]:
calendar_cols = ["year","month","quarter","week_of_year","day_of_week","day_of_month","day_of_year","is_weekend"]
cyclical_cols = ["month_sin","month_cos","dow_sin","dow_cos"]
lag_cols = [c for c in df.columns if c.startswith("units_sold_lag_")]
rolling_cols = [c for c in df.columns if c.startswith("rolling_")]
trend_cols = [c for c in df.columns if c.startswith("demand_")]
price_cols = ["average_unit_price","base_price","discount_pct","price_lag_1","price_change"]
promo_cols = ["promotion_flag","promotion_available","promo_rolling_7"]
product_cols = ["category","sub_category","brand"]
entity_cols = ["region","store_type","store_size_sqft"]
inv_cols = ["ending_inventory","on_order_qty","stockout_flag","historical_doi"]

for name, cols in [
    ("calendar", calendar_cols), ("cyclical", cyclical_cols), ("lag", lag_cols),
    ("rolling", rolling_cols), ("trend", trend_cols), ("price", price_cols),
    ("promo", promo_cols), ("product", product_cols), ("entity", entity_cols),
    ("inventory", inv_cols),
]:
    print(f"{name}: {len(cols)} -> {cols}")

print("\nUCI sample:")
print(df[df.source_dataset=="UCI"].head(3).to_string())
print("SYNTHETIC sample:")
print(df[df.source_dataset=="SYNTHETIC"].head(3).to_string())


calendar: 8 -> ['year', 'month', 'quarter', 'week_of_year', 'day_of_week', 'day_of_month', 'day_of_year', 'is_weekend']
cyclical: 4 -> ['month_sin', 'month_cos', 'dow_sin', 'dow_cos']
lag: 8 -> ['units_sold_lag_1', 'units_sold_lag_2', 'units_sold_lag_3', 'units_sold_lag_7', 'units_sold_lag_14', 'units_sold_lag_21', 'units_sold_lag_28', 'units_sold_lag_30']
rolling: 6 -> ['rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14', 'rolling_mean_30', 'rolling_std_30']
trend: 4 -> ['demand_change_1', 'demand_change_7', 'demand_growth_7', 'demand_growth_30']
price: 5 -> ['average_unit_price', 'base_price', 'discount_pct', 'price_lag_1', 'price_change']
promo: 3 -> ['promotion_flag', 'promotion_available', 'promo_rolling_7']
product: 3 -> ['category', 'sub_category', 'brand']
entity: 3 -> ['region', 'store_type', 'store_size_sqft']
inventory: 4 -> ['ending_inventory', 'on_order_qty', 'stockout_flag', 'historical_doi']

UCI sample:
              date source_dataset entity_id enti

        date source_dataset  entity_id entity_type    product_key     sku_id  units_sold  revenue  average_unit_price  transaction_count  unique_customers  promotion_flag  year  month  quarter  week_of_year  day_of_week  day_of_month  day_of_year  is_weekend  month_sin  month_cos   dow_sin   dow_cos  units_sold_lag_1  units_sold_lag_2  units_sold_lag_3  units_sold_lag_7  units_sold_lag_14  units_sold_lag_21  units_sold_lag_28  units_sold_lag_30  rolling_mean_7  rolling_std_7  rolling_mean_14  rolling_std_14  rolling_mean_30  rolling_std_30  demand_change_1  demand_change_7  demand_growth_7  demand_growth_30  base_price  discount_pct  price_lag_1  price_change  promotion_available  promo_rolling_7 category sub_category     brand region store_type  store_size_sqft  ending_inventory  on_order_qty  stockout_flag  historical_doi  is_holiday  season  split  insufficient_history
0 2022-01-01      SYNTHETIC  STORE_001       STORE  SYN_SKU_00001  SKU_00001          26  3717.74              142.

## 4. Missing-value strategy (no blind zero-fill)

In [4]:
print("insufficient_history rate:", round(100*df['insufficient_history'].mean(), 2), "%")
lag_roll = [c for c in df.columns if "lag_" in c or c.startswith("rolling_") or c.startswith("demand_")]
miss = pd.DataFrame({
    "feature": lag_roll,
    "missing_pct": [round(100*df[c].isna().mean(), 2) for c in lag_roll],
}).sort_values("missing_pct", ascending=False)
print(miss.head(20).to_string(index=False))
print("Strategy: leave warm-up NaNs as NaN; flag with insufficient_history.")


insufficient_history rate: 0.3 %
          feature  missing_pct
units_sold_lag_30         7.47
 demand_growth_30         7.47
units_sold_lag_28         7.04
units_sold_lag_21         5.45
units_sold_lag_14         3.77
  demand_change_7         2.23
 units_sold_lag_7         1.97
  demand_growth_7         1.97
 units_sold_lag_3         0.87
   rolling_std_30         0.59
 units_sold_lag_2         0.59
    rolling_std_7         0.59
  demand_change_1         0.59
   rolling_std_14         0.59
 units_sold_lag_1         0.30
  rolling_mean_14         0.30
   rolling_mean_7         0.30
  rolling_mean_30         0.30
      price_lag_1         0.30
Strategy: leave warm-up NaNs as NaN; flag with insufficient_history.


## 5. Source separation checks

In [5]:
uci = df[df.source_dataset=="UCI"]
syn = df[df.source_dataset=="SYNTHETIC"]
print("UCI rows:", len(uci), "entities:", uci.entity_id.nunique(), "products:", uci.product_key.nunique())
print("SYN rows:", len(syn), "entities:", syn.entity_id.nunique(), "products:", syn.product_key.nunique())
print("UCI promotion_flag all NaN:", bool(uci.promotion_flag.isna().all()))
print("UCI promotion_available all 0:", bool((uci.promotion_available==0).all()))
print("UCI category all NaN:", bool(uci.category.isna().all()))
print("UCI ending_inventory all NaN:", bool(uci.ending_inventory.isna().all()))
print("SYN ending_inventory non-null:", int(syn.ending_inventory.notna().sum()))


UCI rows: 534496 entities: 1 products: 4984
SYN rows: 1461000 entities: 10 products: 100
UCI promotion_flag all NaN: True
UCI promotion_available all 0: True
UCI category all NaN: True
UCI ending_inventory all NaN: True
SYN ending_inventory non-null: 1460000


## 6. Leakage validation suite

In [6]:
result = run_validation()
print("VALIDATION:", result.summary())
assert result.failed == 0, f"Validation failed: {result.failed} checks"


PHASE 6 FEATURE VALIDATION

[1] File Existence & Readability
  [+] Output file exists


  [+] Output file is readable -- 1,995,496 rows x 62 cols

[2] Row Count Validation
  [+] Row count matches input -- output=1,995,496 vs input=1,995,496

[3] Grain Uniqueness


  [+] No duplicate forecasting grain -- duplicates=0

[4] Key Column Nulls
  [+] No null in date -- nulls=0
  [+] No null in source_dataset -- nulls=0
  [+] No null in entity_id -- nulls=0
  [+] No null in product_key -- nulls=0

[5] Date Ordering


  [+] Dates sorted within each grain

[6] Infinite Value Check


  [+] No infinite values in numeric features -- infinite=0

[7] Source Separation
  [+] Two sources present -- sources=['SYNTHETIC', 'UCI']


...

  [+] Source isolation: no shared entity_id across sources -- shared=set()
  [+] Source isolation: no shared product_key across sources -- shared_count=0


  [+] Inventory lag-1: first Synthetic row ending_inventory is NaN -- non-null=0


  [+] Inventory features present after warm-up (Synthetic) -- sample_non_null=500

[9] Time Split Integrity
  [+] Split column exists
  [+] Three split values present -- values=['test', 'train', 'validation']


  [+] SYNTHETIC: train_end < val_start -- train_max=2025-03-13, val_min=2025-03-14
  [+] SYNTHETIC: val_end < test_start -- val_max=2025-08-06, test_min=2025-08-07
  [+] SYNTHETIC: train dates precede validation dates
  [+] SYNTHETIC: validation dates precede test dates


  [+] UCI: train_end < val_start -- train_max=2011-07-13, val_min=2011-07-14
  [+] UCI: val_end < test_start -- val_max=2011-09-25, test_min=2011-09-26
  [+] UCI: train dates precede validation dates
  [+] UCI: validation dates precede test dates

[10] Feature Count & Types
  [+] Feature 'year' exists
  [+] Feature 'month' exists
  [+] Feature 'quarter' exists
  [+] Feature 'week_of_year' exists
  [+] Feature 'day_of_week' exists
  [+] Feature 'day_of_month' exists
  [+] Feature 'day_of_year' exists
  [+] Feature 'is_weekend' exists
  [+] Feature 'month_sin' exists
  [+] Feature 'month_cos' exists
  [+] Feature 'dow_sin' exists
  [+] Feature 'dow_cos' exists
  [+] Feature 'units_sold_lag_1' exists
  [+] Feature 'units_sold_lag_7' exists
  [+] Feature 'units_sold_lag_28' exists
  [+] Feature 'rolling_mean_7' exists
  [+] Feature 'rolling_std_7' exists
  [+] Feature 'rolling_mean_14' exists
  [+] Feature 'rolling_mean_30' exists
  [+] Feature 'demand_change_1' exists
  [+] Feature 'deman

## 7. ML compatibility & final report

In [7]:
compat = get_compatibility_summary()
print(compat)
report_path = write_feature_engineering_report(df, meta, result.summary())
print("Wrote report:", report_path)
print("Phase 6 COMPLETE — STOP before Phase 7.")


{'phase6_output': 'data/processed/features/forecast_features.parquet', 'legacy_consumers': ['src/forecasting.py', 'src/validate_ml_stack.py', 'dashboard/app.py'], 'renames': {'dow_sin': 'sin_day_of_week', 'dow_cos': 'cos_day_of_week', 'month_sin': 'sin_month', 'month_cos': 'cos_month', 'rolling_mean_7': 'units_sold_rolling_mean_7', 'rolling_std_7': 'units_sold_rolling_std_7', 'rolling_mean_14': 'units_sold_rolling_mean_14', 'rolling_mean_30': 'units_sold_rolling_mean_30', 'rolling_std_14': 'units_sold_rolling_std_14', 'rolling_std_30': 'units_sold_rolling_std_30'}, 'legacy_only_features': ['units_sold_ewm_7', 'units_sold_ewm_28'], 'adapter_required': True, 'note': 'Phase 6 does not duplicate the legacy SKU-level feature matrix. Use adapt_phase6_to_legacy_ml() to bridge Phase 6 columns, or build_forecasting_feature_matrix() on a legacy sales frame for the existing ML training path. Phase 7 baselines should consume forecast_features.parquet directly.'}
Wrote report: C:\Users\SURAG\Docume